# Substitution-Aware Entropy Pipeline (Notebook)

This notebook is a cell-by-cell version of `entropy_pipeline.py`.
Run the cells from top to bottom so each stage is executed in order.

In [ ]:
import csv
import json
import math
import sys
from collections import Counter
from pathlib import Path

AA_ORDER = list("ARNDCQEGHILKMFPSTWYV")
AA_SET = set(AA_ORDER)

BLOSUM62 = {
    "A": {"A":4,"R":-1,"N":-2,"D":-2,"C":0,"Q":-1,"E":-1,"G":0,"H":-2,"I":-1,"L":-1,"K":-1,"M":-1,"F":-2,"P":-1,"S":1,"T":0,"W":-3,"Y":-2,"V":0},
    "R": {"A":-1,"R":5,"N":0,"D":-2,"C":-3,"Q":1,"E":0,"G":-2,"H":0,"I":-3,"L":-2,"K":2,"M":-1,"F":-3,"P":-2,"S":-1,"T":-1,"W":-3,"Y":-2,"V":-3},
    "N": {"A":-2,"R":0,"N":6,"D":1,"C":-3,"Q":0,"E":0,"G":0,"H":1,"I":-3,"L":-3,"K":0,"M":-2,"F":-3,"P":-2,"S":1,"T":0,"W":-4,"Y":-2,"V":-3},
    "D": {"A":-2,"R":-2,"N":1,"D":6,"C":-3,"Q":0,"E":2,"G":-1,"H":-1,"I":-3,"L":-4,"K":-1,"M":-3,"F":-3,"P":-1,"S":0,"T":-1,"W":-4,"Y":-3,"V":-3},
    "C": {"A":0,"R":-3,"N":-3,"D":-3,"C":9,"Q":-3,"E":-4,"G":-3,"H":-3,"I":-1,"L":-1,"K":-3,"M":-1,"F":-2,"P":-3,"S":-1,"T":-1,"W":-2,"Y":-2,"V":-1},
    "Q": {"A":-1,"R":1,"N":0,"D":0,"C":-3,"Q":5,"E":2,"G":-2,"H":0,"I":-3,"L":-2,"K":1,"M":0,"F":-3,"P":-1,"S":0,"T":-1,"W":-2,"Y":-1,"V":-2},
    "E": {"A":-1,"R":0,"N":0,"D":2,"C":-4,"Q":2,"E":5,"G":-2,"H":0,"I":-3,"L":-3,"K":1,"M":-2,"F":-3,"P":-1,"S":0,"T":-1,"W":-3,"Y":-2,"V":-2},
    "G": {"A":0,"R":-2,"N":0,"D":-1,"C":-3,"Q":-2,"E":-2,"G":6,"H":-2,"I":-4,"L":-4,"K":-2,"M":-3,"F":-3,"P":-2,"S":0,"T":-2,"W":-2,"Y":-3,"V":-3},
    "H": {"A":-2,"R":0,"N":1,"D":-1,"C":-3,"Q":0,"E":0,"G":-2,"H":8,"I":-3,"L":-3,"K":-1,"M":-2,"F":-1,"P":-2,"S":-1,"T":-2,"W":-2,"Y":2,"V":-3},
    "I": {"A":-1,"R":-3,"N":-3,"D":-3,"C":-1,"Q":-3,"E":-3,"G":-4,"H":-3,"I":4,"L":2,"K":-3,"M":1,"F":0,"P":-3,"S":-2,"T":-1,"W":-3,"Y":-1,"V":3},
    "L": {"A":-1,"R":-2,"N":-3,"D":-4,"C":-1,"Q":-2,"E":-3,"G":-4,"H":-3,"I":2,"L":4,"K":-2,"M":2,"F":0,"P":-3,"S":-2,"T":-1,"W":-2,"Y":-1,"V":1},
    "K": {"A":-1,"R":2,"N":0,"D":-1,"C":-3,"Q":1,"E":1,"G":-2,"H":-1,"I":-3,"L":-2,"K":5,"M":-1,"F":-3,"P":-1,"S":0,"T":-1,"W":-3,"Y":-2,"V":-2},
    "M": {"A":-1,"R":-1,"N":-2,"D":-3,"C":-1,"Q":0,"E":-2,"G":-3,"H":-2,"I":1,"L":2,"K":-1,"M":5,"F":0,"P":-2,"S":-1,"T":-1,"W":-1,"Y":-1,"V":1},
    "F": {"A":-2,"R":-3,"N":-3,"D":-3,"C":-2,"Q":-3,"E":-3,"G":-3,"H":-1,"I":0,"L":0,"K":-3,"M":0,"F":6,"P":-4,"S":-2,"T":-2,"W":1,"Y":3,"V":-1},
    "P": {"A":-1,"R":-2,"N":-2,"D":-1,"C":-3,"Q":-1,"E":-1,"G":-2,"H":-2,"I":-3,"L":-3,"K":-1,"M":-2,"F":-4,"P":7,"S":-1,"T":-1,"W":-4,"Y":-3,"V":-2},
    "S": {"A":1,"R":-1,"N":1,"D":0,"C":-1,"Q":0,"E":0,"G":0,"H":-1,"I":-2,"L":-2,"K":0,"M":-1,"F":-2,"P":-1,"S":4,"T":1,"W":-3,"Y":-2,"V":-2},
    "T": {"A":0,"R":-1,"N":0,"D":-1,"C":-1,"Q":-1,"E":-1,"G":-2,"H":-2,"I":-1,"L":-1,"K":-1,"M":-1,"F":-2,"P":-1,"S":1,"T":5,"W":-2,"Y":-2,"V":0},
    "W": {"A":-3,"R":-3,"N":-4,"D":-4,"C":-2,"Q":-2,"E":-3,"G":-2,"H":-2,"I":-3,"L":-2,"K":-3,"M":-1,"F":1,"P":-4,"S":-3,"T":-2,"W":11,"Y":2,"V":-3},
    "Y": {"A":-2,"R":-2,"N":-2,"D":-3,"C":-2,"Q":-1,"E":-2,"G":-3,"H":2,"I":-1,"L":-1,"K":-2,"M":-1,"F":3,"P":-3,"S":-2,"T":-2,"W":2,"Y":7,"V":-1},
    "V": {"A":0,"R":-3,"N":-3,"D":-3,"C":-1,"Q":-2,"E":-2,"G":-3,"H":-3,"I":3,"L":1,"K":-2,"M":1,"F":-1,"P":-2,"S":-2,"T":0,"W":-3,"Y":-1,"V":4},
}

MAX_ENTROPY = math.log2(20)
GAP_CHARS = set("-. ")

In [ ]:
def read_fasta(path: str) -> list[tuple[str, str]]:
    records, header, seq_parts = [], None, []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip()
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts).upper()))
                header, seq_parts = line[1:], []
            else:
                seq_parts.append(line)
    if header is not None:
        records.append((header, "".join(seq_parts).upper()))
    return records


def split_combined(records: list[tuple[str, str]], design_prefix: str) -> tuple[list[tuple[str, str]], list[tuple[str, str]]]:
    prefix = design_prefix.lower()
    natural, designed = [], []
    for header, seq in records:
        if header.lower().startswith(prefix):
            designed.append((header, seq))
        else:
            natural.append((header, seq))
    return natural, designed


def blosum62_softmax(aa: str) -> list[float]:
    row = BLOSUM62[aa]
    scores = [row[b] for b in AA_ORDER]
    max_s = max(scores)
    exps = [math.exp(s - max_s) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]


def column_blosum62_distribution(column_aas: list[str]) -> list[float]:
    valid = [aa for aa in column_aas if aa in AA_SET]
    if not valid:
        return [1.0 / 20] * 20
    counts = Counter(valid)
    total = sum(counts.values())
    mixed = [0.0] * 20
    for aa, cnt in counts.items():
        w = cnt / total
        probs = blosum62_softmax(aa)
        mixed = [m + w * p for m, p in zip(mixed, probs)]
    return mixed


def shannon_entropy(probs: list[float]) -> float:
    return -sum(p * math.log2(p) for p in probs if p > 1e-12)

In [ ]:
def compute_natural_profile(records: list[tuple[str, str]]) -> list[dict]:
    if not records:
        raise ValueError("No natural sequences found.")

    aln_len = len(records[0][1])
    seqs = [s for _, s in records]
    profile = []

    for col in range(aln_len):
        col_chars = [s[col] for s in seqs if col < len(s)]
        gap_count = sum(1 for c in col_chars if c in GAP_CHARS)
        gap_frac = gap_count / len(col_chars) if col_chars else 1.0
        col_aas = [c for c in col_chars if c not in GAP_CHARS]

        freq_counts = Counter(c for c in col_aas if c in AA_SET)
        n_valid = sum(freq_counts.values())
        freq_dist = (
            {aa: freq_counts.get(aa, 0) / n_valid for aa in AA_ORDER}
            if n_valid else {aa: 0.05 for aa in AA_ORDER}
        )

        mixed_dist = column_blosum62_distribution(col_aas)
        h_val = shannon_entropy(mixed_dist)
        norm_h = h_val / MAX_ENTROPY

        profile.append({
            "col": col,
            "position": col + 1,
            "gap_frac": round(gap_frac, 6),
            "n_seqs": len(col_aas),
            "freq_dist": freq_dist,
            "blosum_dist": {aa: round(p, 8) for aa, p in zip(AA_ORDER, mixed_dist)},
            "H_bits": round(h_val, 8),
            "norm_H": round(norm_h, 8),
        })

    return profile


def save_natural_profile(profile: list[dict], prefix: str) -> str:
    json_path = f"{prefix}_profile.json"
    with open(json_path, "w") as f:
        json.dump(profile, f, indent=2)
    print(f"Natural entropy profile (JSON) -> {json_path}")

    csv_path = f"{prefix}_profile.csv"
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(
            ["position", "gap_frac", "n_seqs", "H_bits", "norm_H"] +
            [f"freq_{aa}" for aa in AA_ORDER] +
            [f"blosum_{aa}" for aa in AA_ORDER]
        )
        for c in profile:
            w.writerow(
                [c["position"], c["gap_frac"], c["n_seqs"], c["H_bits"], c["norm_H"]] +
                [round(c["freq_dist"].get(aa, 0), 6) for aa in AA_ORDER] +
                [round(c["blosum_dist"].get(aa, 0), 8) for aa in AA_ORDER]
            )
    print(f"Natural entropy profile (CSV)  -> {csv_path}")
    return json_path

In [ ]:
def score_designed_sequence(seq: str, profile: list[dict]) -> list[dict]:
    scores = []
    for col, prof in enumerate(profile):
        if col >= len(seq):
            break
        aa = seq[col]
        if aa in GAP_CHARS:
            continue

        aa = aa.upper()
        norm_h = prof["norm_H"]

        p_aa = max(prof["blosum_dist"].get(aa, 1e-8), 1e-8)
        neg_log_p = -math.log2(p_aa)
        norm_neg_logp = neg_log_p / MAX_ENTROPY
        combined = (norm_h + norm_neg_logp) / 2.0

        scores.append({
            "col": col,
            "position": col + 1,
            "aa": aa,
            "gap_frac_natural": prof["gap_frac"],
            "H_bits": prof["H_bits"],
            "penalty_A_entropy_lookup": round(norm_h, 6),
            "penalty_B_distribution_dev": round(norm_neg_logp, 6),
            "combined_penalty": round(combined, 6),
            "p_aa_in_natural": round(p_aa, 8),
        })
    return scores


def aggregate_scores(scores: list[dict]) -> dict:
    if not scores:
        return {}
    keys = [
        "penalty_A_entropy_lookup",
        "penalty_B_distribution_dev",
        "combined_penalty",
    ]
    return {
        "n_positions": len(scores),
        **{f"mean_{k}": round(sum(s[k] for s in scores) / len(scores), 6) for k in keys},
        **{f"sum_{k}": round(sum(s[k] for s in scores), 6) for k in keys},
        **{f"max_{k}": round(max(s[k] for s in scores), 6) for k in keys},
        **{f"min_{k}": round(min(s[k] for s in scores), 6) for k in keys},
    }


def score_all_designed(records: list[tuple[str, str]], profile: list[dict], prefix: str) -> list[dict]:
    all_results = []

    for header, seq in records:
        scores = score_designed_sequence(seq, profile)
        agg = aggregate_scores(scores)
        all_results.append({
            "header": header,
            "sequence": seq.replace("-", "").replace(".", ""),
            "aggregate": agg,
            "per_position": scores,
        })

    json_path = f"{prefix}_designed_scores.json"
    with open(json_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Designed scores (JSON) -> {json_path}")

    if all_results:
        sum_csv = f"{prefix}_designed_summary.csv"
        summary_rows = [{"sequence": r["header"], **r["aggregate"]} for r in all_results]
        with open(sum_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
            w.writeheader()
            w.writerows(summary_rows)
        print(f"Designed summary CSV -> {sum_csv}")

    pos_csv = f"{prefix}_designed_per_position.csv"
    with open(pos_csv, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "sequence", "col", "position", "aa",
            "H_bits", "gap_frac_natural",
            "penalty_A_entropy_lookup",
            "penalty_B_distribution_dev",
            "combined_penalty",
            "p_aa_in_natural",
        ])
        for res in all_results:
            for s in res["per_position"]:
                w.writerow([
                    res["header"],
                    s["col"], s["position"], s["aa"],
                    s["H_bits"], s["gap_frac_natural"],
                    s["penalty_A_entropy_lookup"],
                    s["penalty_B_distribution_dev"],
                    s["combined_penalty"],
                    s["p_aa_in_natural"],
                ])
    print(f"Designed per-position CSV -> {pos_csv}")

    return all_results

In [ ]:
def _style_axes(axes):
    for ax in axes:
        ax.set_facecolor("#111827")
        for sp in ax.spines.values():
            sp.set_edgecolor("#374151")
        ax.tick_params(colors="#9ca3af", labelsize=8)
        ax.yaxis.label.set_color("#9ca3af")
        ax.xaxis.label.set_color("#9ca3af")
        ax.title.set_color("#e2e8f0")


def _entropy_cmap():
    import matplotlib.colors as mc
    return mc.LinearSegmentedColormap.from_list(
        "ep", [(0,"#14b8a6"),(0.33,"#eab308"),(0.66,"#f97316"),(1,"#dc2626")]
    )


def plot_natural_profile(profile: list[dict], prefix: str):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Install matplotlib to enable plots.", file=sys.stderr)
        return

    positions = [c["position"] for c in profile]
    norm_h = [c["norm_H"] for c in profile]
    h_bits = [c["H_bits"] for c in profile]
    gap_frac = [c["gap_frac"] for c in profile]
    cmap = _entropy_cmap()

    fig, axes = plt.subplots(3, 1, figsize=(max(12, len(positions) * 0.12), 10), facecolor="#0b0f1a")
    fig.suptitle("Natural Sequence Entropy Profile (BLOSUM62-Weighted)", color="#e2e8f0", fontsize=12, fontweight="bold")
    _style_axes(axes)

    axes[0].bar(positions, norm_h, color=[cmap(v) for v in norm_h], width=1.0)
    axes[0].axhline(sum(norm_h)/len(norm_h), color="#6b7280", lw=1, ls="--", label=f"mean = {sum(norm_h)/len(norm_h):.3f}")
    axes[0].set_ylabel("Normalized Entropy")
    axes[0].set_title("Per-Column Normalized Entropy")
    axes[0].legend(fontsize=8, labelcolor="#9ca3af", facecolor="#1f2937", edgecolor="#374151")

    axes[1].plot(positions, h_bits, color="#14b8a6", lw=1.2)
    axes[1].axhline(MAX_ENTROPY, color="#ef4444", lw=1, ls=":", label=f"max H = {MAX_ENTROPY:.3f} bits")
    axes[1].axhline(sum(h_bits)/len(h_bits), color="#6b7280", lw=1, ls="--", label=f"mean H = {sum(h_bits)/len(h_bits):.3f} bits")
    axes[1].set_ylabel("Shannon Entropy (bits)")
    axes[1].set_title("Per-Column Shannon Entropy")
    axes[1].legend(fontsize=8, labelcolor="#9ca3af", facecolor="#1f2937", edgecolor="#374151")

    axes[2].fill_between(positions, gap_frac, color="#374151", alpha=0.9)
    axes[2].set_ylabel("Gap Fraction")
    axes[2].set_title("Column Gap Fraction")
    axes[2].set_xlabel("Alignment Column")
    axes[2].set_ylim(0, 1)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    out = f"{prefix}_natural_profile.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor="#0b0f1a")
    plt.close()
    print(f"Natural profile plot -> {out}")

In [ ]:
def print_natural_summary(profile: list[dict], n_seqs: int):
    norm_h = [c["norm_H"] for c in profile]
    h_bits = [c["H_bits"] for c in profile]
    print("\nStage 1 - Natural Entropy Profile")
    print("-" * 55)
    print(f"Sequences analyzed : {n_seqs}")
    print(f"Alignment columns  : {len(profile)}")
    print(f"Mean norm entropy  : {sum(norm_h)/len(norm_h):.6f}")
    print(f"Max norm entropy   : {max(norm_h):.6f}")
    print(f"Min norm entropy   : {min(norm_h):.6f}")
    print(f"Mean H (bits)      : {sum(h_bits)/len(h_bits):.6f}")


def print_designed_summary(all_results: list[dict]):
    print("\nStage 2 - Designed Sequence Penalties")
    print("-" * 68)
    print(f"{'Sequence':<36} {'Pen-A':>8} {'Pen-B':>8} {'Combined':>9}")
    print(f"{'-'*35} {'-'*7} {'-'*7} {'-'*8}")
    for r in all_results:
        agg = r["aggregate"]
        name = r["header"][:35]
        pA = agg.get("mean_penalty_A_entropy_lookup", 0)
        pB = agg.get("mean_penalty_B_distribution_dev", 0)
        pC = agg.get("mean_combined_penalty", 0)
        print(f"{name:<36} {pA:>8.4f} {pB:>8.4f} {pC:>9.4f}")


def print_split_report(natural, designed, prefix):
    print(f"Sequences matching prefix '{prefix}' -> designed: {len(designed)}")
    print(f"Remaining sequences (natural): {len(natural)}")
    if not designed:
        raise ValueError(f"No designed sequences found with prefix '{prefix}'.")

## Configure Inputs
Set your file names and designed-sequence prefix here before running Stage 1 and Stage 2 cells.

In [ ]:
# Use the files currently in this folder by default.
natural_fasta = "22_cut_msa.fasta"
combined_fasta = "all_aligned.fasta"
design_prefix = "cuti1cex"
output_prefix = "results"
make_plots = True

print("Working directory:", Path.cwd())
print("natural_fasta exists:", Path(natural_fasta).exists())
print("combined_fasta exists:", Path(combined_fasta).exists())

## Run Stage 1 (Natural Profile)

In [ ]:
# Stage 1 execution with fallback imports for fresh/restarted kernels.

if (
    "read_fasta" not in globals()
    or "compute_natural_profile" not in globals()
    or "print_natural_summary" not in globals()
    or "save_natural_profile" not in globals()
    or "plot_natural_profile" not in globals()
):
    from entropy_pipeline import (
        read_fasta,
        compute_natural_profile,
        print_natural_summary,
        save_natural_profile,
        plot_natural_profile,
    )

if "natural_fasta" not in globals():
    natural_fasta = "22_cut_msa.fasta"
if "output_prefix" not in globals():
    output_prefix = "results"
if "make_plots" not in globals():
    make_plots = True

natural_records = read_fasta(natural_fasta)
print(f"Loaded {len(natural_records)} natural sequences")
profile = compute_natural_profile(natural_records)
print_natural_summary(profile, len(natural_records))
profile_json = save_natural_profile(profile, output_prefix)
if make_plots:
    plot_natural_profile(profile, output_prefix)

profile_json

## Run Stage 2 (Designed Scoring)

In [ ]:
all_records = read_fasta(combined_fasta)
natural_part, designed_part = split_combined(all_records, design_prefix)
print_split_report(natural_part, designed_part, design_prefix)
all_results = score_all_designed(designed_part, profile, output_prefix)
print_designed_summary(all_results)

len(all_results)

In [ ]:
# Find entropy penalties for all natural sequences using the Stage 1 profile.

import csv
import json
from pathlib import Path

# Defaults in case input/config cells were not run yet.
if "output_prefix" not in globals():
    output_prefix = "results"
if "combined_fasta" not in globals():
    combined_fasta = "all_aligned.fasta"
if "design_prefix" not in globals():
    design_prefix = "cuti1cex"

# Fallback import if notebook function-definition cells were not executed.
if "read_fasta" not in globals() or "split_combined" not in globals():
    from entropy_pipeline import (
        read_fasta,
        split_combined,
        score_designed_sequence,
        aggregate_scores,
    )

# Load profile if Stage 1 cell was not run in this kernel session.
if "profile" not in globals():
    profile_json = f"{output_prefix}_profile.json"
    if not Path(profile_json).exists():
        raise FileNotFoundError(
            f"Profile not found: {profile_json}. Run Stage 1 cell first or place the file in this folder."
        )
    with open(profile_json) as f:
        profile = json.load(f)

if "natural_part" not in globals():
    all_records = read_fasta(combined_fasta)
    natural_part, designed_part = split_combined(all_records, design_prefix)

natural_penalty_results = []
for header, seq in natural_part:
    per_pos = score_designed_sequence(seq, profile)
    agg = aggregate_scores(per_pos)
    natural_penalty_results.append({
        "header": header,
        "aggregate": agg,
        "per_position": per_pos,
    })

# Save outputs
natural_json = f"{output_prefix}_natural_scores.json"
with open(natural_json, "w") as f:
    json.dump(natural_penalty_results, f, indent=2)

natural_summary_csv = f"{output_prefix}_natural_summary.csv"
rows = []
if natural_penalty_results:
    rows = [{"sequence": r["header"], **r["aggregate"]} for r in natural_penalty_results]
    with open(natural_summary_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

print(f"Scored natural sequences: {len(natural_penalty_results)}")
print(f"Natural scores JSON -> {natural_json}")
print(f"Natural summary CSV -> {natural_summary_csv}")

rows[:5]

In [ ]:
# Score the added sequence from a MAFFT-aligned FASTA that already contains natural sequences + the new sequence.

import csv
import json
from pathlib import Path

# Provide your aligned FASTA here. It must contain the natural sequences and the added sequence in the same alignment.
aligned_fasta = "22_cut_msa_with_cuti_07475.fasta"

# Identify the added sequence by exact header first; if not found, the last sequence is used as fallback.
added_sequence_header = "manual_cuti_07475"

if "GAP_CHARS" not in globals():
    GAP_CHARS = set("-. ")

# Fallback imports if earlier definition cells were not run in this kernel.
if (
    "read_fasta" not in globals()
    or "compute_natural_profile" not in globals()
    or "save_natural_profile" not in globals()
    or "score_designed_sequence" not in globals()
    or "aggregate_scores" not in globals()
):
    from entropy_pipeline import (
        read_fasta,
        compute_natural_profile,
        save_natural_profile,
        score_designed_sequence,
        aggregate_scores,
    )

records = read_fasta(aligned_fasta)
if not records:
    raise ValueError(f"No sequences found in {aligned_fasta}")

aln_len = len(records[0][1])
if any(len(seq) != aln_len for _, seq in records):
    raise ValueError("Aligned FASTA contains inconsistent sequence lengths.")

# Pick the added sequence.
added_record = None
for header, seq in records:
    if header == added_sequence_header or header.startswith(added_sequence_header):
        added_record = (header, seq)
        break

if added_record is None:
    added_record = records[-1]
    print(f"Added sequence header '{added_sequence_header}' not found; using last record: {added_record[0]}")

added_header, added_seq = added_record

# Use all other sequences as the natural background for the entropy profile.
natural_records = [(header, seq) for header, seq in records if header != added_header]
if not natural_records:
    raise ValueError("No natural sequences remain after excluding the added sequence.")

profile = compute_natural_profile(natural_records)
profile_json = save_natural_profile(profile, "results_added_sequence")

per_pos_scores = score_designed_sequence(added_seq, profile)
agg_scores = aggregate_scores(per_pos_scores)

print("\nAdded Sequence Penalty Summary")
print("-" * 68)
print(f"{'Sequence':<36} {'Pen-A':>8} {'Pen-B':>8} {'Combined':>9}")
print(f"{'-'*35} {'-'*7} {'-'*7} {'-'*8}")
name = added_header[:35]
pA = agg_scores.get("mean_penalty_A_entropy_lookup", 0)
pB = agg_scores.get("mean_penalty_B_distribution_dev", 0)
pC = agg_scores.get("mean_combined_penalty", 0)
print(f"{name:<36} {pA:>8.4f} {pB:>8.4f} {pC:>9.4f}")
print(f"Alignment columns: {aln_len} | Positions scored: {agg_scores.get('n_positions', 0)}")
print(f"Natural sequences used for profile: {len(natural_records)}")

# Save outputs
manual_json = f"results_{added_header}_scores.json"
with open(manual_json, "w") as f:
    json.dump(
        {
            "name": added_header,
            "aligned_fasta": aligned_fasta,
            "sequence_aligned_used_for_scoring": added_seq,
            "aggregate": agg_scores,
            "per_position": per_pos_scores,
        },
        f,
        indent=2,
    )

manual_csv = f"results_{added_header}_per_position.csv"
if per_pos_scores:
    with open(manual_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(per_pos_scores[0].keys()))
        w.writeheader()
        w.writerows(per_pos_scores)

print(f"Saved JSON -> {manual_json}")
print(f"Saved per-position CSV -> {manual_csv}")

per_pos_scores[:10]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    import logomaker
except ImportError as exc:
    raise ImportError(
        "Please install logomaker first: pip install logomaker"
    ) from exc

# Input from Stage 1 output
profile_csv = f"{output_prefix}_profile.csv"

# Optional global filter (1-based). Keep as None to include all positions first.
start_pos = None
end_pos = None

# Number of alignment positions per image.
window_size = 50


def _build_matrix(df: pd.DataFrame, prefix: str, aa_order: list[str]) -> pd.DataFrame:
    cols = [f"{prefix}{aa}" for aa in aa_order]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns for {prefix}: {missing[:5]} ...")

    mat = df[cols].copy()
    mat.columns = aa_order
    mat.index = df["position"].astype(int)

    # Normalize rows to sum to 1 for stable logo plotting.
    row_sums = mat.sum(axis=1).replace(0, 1.0)
    mat = mat.div(row_sums, axis=0)
    return mat


def _plot_window(window_df: pd.DataFrame, out_path: str) -> None:
    freq_matrix = _build_matrix(window_df, "freq_", AA_ORDER)
    blosum_matrix = _build_matrix(window_df, "blosum_", AA_ORDER)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    logomaker.Logo(freq_matrix, ax=axes[0], color_scheme="chemistry")
    axes[0].set_title("Sequence Logo from Observed Frequencies (freq_*)")
    axes[0].set_ylabel("Probability")
    axes[0].set_ylim(0, 1)

    logomaker.Logo(blosum_matrix, ax=axes[1], color_scheme="chemistry")
    axes[1].set_title("Substitution-Aware Logo (BLOSUM-smoothed, blosum_*)")
    axes[1].set_ylabel("Probability")
    axes[1].set_xlabel("Alignment Position")
    axes[1].set_ylim(0, 1)

    positions = window_df["position"].tolist()
    tick_step = max(1, len(positions) // 8)
    axes[1].set_xticks(positions[::tick_step])

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


df = pd.read_csv(profile_csv)

if start_pos is not None and end_pos is not None:
    if start_pos > end_pos:
        raise ValueError("start_pos must be <= end_pos")
    df = df[(df["position"] >= start_pos) & (df["position"] <= end_pos)].copy()

if df.empty:
    raise ValueError("No rows selected. Check start_pos/end_pos and profile CSV content.")

df = df.sort_values("position").reset_index(drop=True)
positions = df["position"].tolist()

saved_files = []
for i in range(0, len(positions), window_size):
    chunk_positions = positions[i:i + window_size]
    pmin, pmax = int(chunk_positions[0]), int(chunk_positions[-1])
    chunk_df = df[df["position"].isin(chunk_positions)].copy()

    out_file = f"{output_prefix}_sequence_logos_{pmin:03d}_{pmax:03d}.png"
    _plot_window(chunk_df, out_file)
    saved_files.append(out_file)

print(f"Created {len(saved_files)} logo image(s), {window_size} positions per image:")
for f in saved_files:
    print(" -", f)